# Simplified Chinese support and reach on Steam

DATA2010 final project. **This notebook is the whole project**: it goes from the two raw
input files (snapshots A and C of the Steam Games Dataset) to the analysis tables, and the
analysis (EDA, SQ1, SQ2, SQ3) is added below in Part B.

**How to run from scratch**
1. Python 3.12, then `pip install -r requirements.txt` (exact versions; GitHub Codespaces
   does this for you).
2. Open this notebook and *Run All*, or from the repository root:
   `jupyter nbconvert --to notebook --execute steam_chinese_analysis.ipynb --output executed.ipynb`

**Input**: only the two original CSV files, snapshot A (2026-06-20) and snapshot C
(2024-01-11) of the Steam Games Dataset. Section A1 downloads them at pinned versions into
`data/raw/` (if the Hugging Face history is ever unavailable, take them from the GitHub
release `raw-snapshots-v1` and unzip them into `data/raw/`). Nothing is edited by hand, and
two runs give byte-identical CSV files.

**Output of Part A**: the two cleaned tables, each saved whole as
`data/processed/csv/pcd_<table>.csv` (to read in code with `pd.read_csv`) and
`data/processed/xlsx/pcd_<table>.xlsx` (to open and look at).

| Table | One row is | Used for |
|---|---|---|
| `pcd_sq1_games` | one game in snapshot A (2026), after cleaning | EDA, SQ1 |
| `pcd_sq2_games` | one game present in both snapshots, with no Chinese in January 2024 | SQ2 |

SQ3 needs no table of its own: it combines the results of SQ1 and SQ2.

Every fixed choice is in **Settings and decisions** with its reason. The first line of each
code cell says what it does (LOAD DATA, DATA CLEANING, SELECT ROWS, AGGREGATE, SAVE, CHECKS).
Problems found in the real data are marked `DATA ISSUE`.

## 0. Setup

In [ ]:
# SETUP: libraries and folders
import ast
import csv
import re
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# The repository root is the first folder (from here upwards) that holds requirements.txt,
# so the notebook works wherever Jupyter starts.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
RAW = ROOT / "data" / "raw"                       # downloaded inputs (not committed)
PROCESSED_CSV = ROOT / "data" / "processed" / "csv"
PROCESSED_XLSX = ROOT / "data" / "processed" / "xlsx"
PREFIX = "pcd_"
print("repository root:", ROOT)

## Settings and decisions

All fixed choices of the project, each with the reason for it. Change them only with the
team leader's sign-off and add a dated line to `docs/decisions.md`.

In [ ]:
# SETTINGS: every fixed choice of the project, with the reason for it

# Seed for every random step (bootstrap), so results are identical on every run.
SEED = 2010

# =============================================================================
# DATA SOURCE
# =============================================================================
# Steam Games Dataset by FronkonGames on Hugging Face. We read two versions of the same
# file from the repository history, pinned by commit hash, so the downloaded bytes never
# change even when the author keeps updating the dataset.
HF_DATASET = "FronkonGames/steam-games-dataset"
HF_FILENAME = "games.csv"

# Snapshot A ("current"): the latest commit that updated games.csv (2026-06-20).
# Later commits (up to 2026-11-08) only update games.json (about 1 GB); kept on the CSV
# so that the pipeline stays fast and simple. Decision: team leader, 2026-11-12.
REVISION_A = "52ba62abfa707a2fa31f4d147c2777a9d9fbd2da"
SNAPSHOT_DATE_A = "2026-06-20"

# Snapshot C ("before"): commit "Update 12-01-2024." of 2024-01-11, the January 2024
# version of the catalogue. Used only by SQ2, to see which games added Chinese later.
REVISION_C = "aefdeb1ff1a739b92ac5338682cc2610a0328505"
SNAPSHOT_DATE_C = "2024-01-11"

# Raw column name -> English snake_case name used everywhere in this repo.
# DATA ISSUE handled in read_header() (section A1): in snapshot A the header merges two names into
# "DiscountDLC count" while every row holds two values, which shifts all later columns.
RAW_TO_CLEAN = {
    "AppID": "app_id",
    "Name": "name",
    "Release date": "release_date",
    "Price": "price",
    "Supported languages": "supported_languages",
    "Full audio languages": "full_audio_languages",
    "Positive": "positive",
    "Negative": "negative",
    "Developers": "developers",
    "Publishers": "publishers",
    "Genres": "genres",
    "Tags": "tags",
}
# Columns that exist in only some snapshots; filled with "0" where missing.
# DATA ISSUE: snapshot A was taken during a Steam sale, so "Price" is the sale price that
# day (Hades: 7.49 = 24.99 at -70%). "Discount" lets us recover the list price. Snapshot C
# has no "Discount" column; its prices are used as listed.
OPTIONAL_RAW_TO_CLEAN = {
    "Discount": "discount_pct",
}

# =============================================================================
# LANGUAGES
# =============================================================================
ZH_SIMPLIFIED = "Simplified Chinese"
ZH_TRADITIONAL = "Traditional Chinese"
ENGLISH = "English"

# "Major" languages. In SQ2 a CONTROL game must not have added Chinese AND must not have
# added any of these either: a game that added German and French in the same period got
# a localisation push of its own, so it is not a clean "did nothing" comparison.
# Chosen as the most common non-English Steam languages other than Chinese.
MAJOR_LANGS = [
    "German", "French", "Spanish - Spain", "Japanese", "Russian",
    "Korean", "Portuguese - Brazil",
]

# =============================================================================
# CHINESE GAMES: ALL are removed, large or small (decision: team leader, 2026-11-12)
# =============================================================================
# Why: a game made in China has Chinese and Chinese players by default. Comparing it with
# Western games would mix "made for China" with "translated into Chinese".
# The dataset has no "country" column, so a game is treated as Chinese if ANY rule matches
# (the rule that matched is written to pcd_cn_excluded so it can be checked by hand):
#   1. Chinese text or voice-over WITHOUT English (made for the Chinese market).
#   2-4. EVERY developer of the game is Chinese, by one of these signs:
#   2. the developer name is written in Chinese characters;
#   3. the developer name contains a mainland Chinese city (Chinese companies are
#      registered as "Shanghai ... Network Technology Co., Ltd." and similar);
#   4. the developer is a well-known Chinese company (list below).
# Rule 1 alone is not enough: Black Myth: Wukong (Game Science) ships English, so only
# rule 4 catches it. Chinese voice-over alone is NOT a rule: many Western games such as
# Cyberpunk 2077 or The Witcher 3 have it (an earlier rule removed 4,233 such games).
# Rules 2-4 look at DEVELOPERS only and need ALL of them to be Chinese: publishers are often
# regional partners (bilibili publishes the Swedish Core Keeper in China) and big Western
# games list Chinese support studios (Far Cry: "Ubisoft Montreal, ..., Ubisoft Shanghai").
CN_CITIES = [
    "Beijing", "Shanghai", "Shenzhen", "Guangzhou", "Hangzhou", "Chengdu", "Nanjing",
    "Wuhan", "Xiamen", "Suzhou", "Chongqing", "Tianjin", "Changsha", "Fuzhou", "Hefei",
    "Jinan", "Qingdao", "Dalian", "Zhuhai", "Dongguan", "Xi'an",
]
CN_COMPANIES = [
    "Tencent", "NetEase", "miHoYo", "HoYoverse", "Game Science", "Perfect World",
    "X.D. Network", "XD Inc", "bilibili", "Gamera Games", "Hypergryph", "Lilith Games",
    "Seasun", "Kingsoft", "Papergames", "Paper Games", "Yostar", "Pathea", "NExT Studios",
    "TiMi Studio", "Leiting", "Giant Network", "Snail Games", "Shanda", "CMGE", "Zulong",
    "Hero Games", "Youthcat", "24 Entertainment",
]
# Legal suffixes left over when a developer list is split on commas ("FromSoftware, Inc.").
COMPANY_SUFFIXES = ["inc", "ltd", "llc", "co", "corp", "limited", "gmbh", "plc", "sa", "srl", "co. ltd"]

# =============================================================================
# GENRE GROUPS
# =============================================================================
# Six groups, from the framework of 2026-11-03. A game gets the group of its FIRST tag that
# appears in this map. Steam lists user tags by number of votes, so the first matching tag
# is the one players use most. Tags such as "Indie" or "Simulation" are skipped: they say
# nothing about how much text a game has. Games with no matching tag are dropped.
TAG_TO_GENRE = {
    "Visual Novel": "Visual Novel",
    "RPG": "RPG", "JRPG": "RPG", "CRPG": "RPG", "Action RPG": "RPG", "Party-Based RPG": "RPG",
    "Strategy": "Strategy", "Turn-Based Strategy": "Strategy", "RTS": "Strategy",
    "Grand Strategy": "Strategy", "4X": "Strategy", "Tower Defense": "Strategy",
    "Action": "Action", "Shooter": "Action", "FPS": "Action", "Platformer": "Action",
    "Hack and Slash": "Action", "Fighting": "Action",
    "Puzzle": "Puzzle", "Hidden Object": "Puzzle",
    "Casual": "Casual",
}

# =============================================================================
# BINS
# =============================================================================
# Price tiers in USD, on the LIST price (see OPTIONAL_RAW_TO_CLEAN). Five tiers from the
# framework: price stands in for budget and studio size.
PRICE_BINS = [-0.001, 0.0, 4.999, 9.999, 19.999, float("inf")]
PRICE_LABELS = ["free", "lt5", "5to10", "10to20", "gt20"]
# Number of languages other than English and Chinese: games with Chinese are usually
# translated as a package, so this is held fixed when comparing.
OTHER_LANG_BINS = [-1, 0, 4, 10_000]
OTHER_LANG_LABELS = ["0", "1-4", "5+"]
# Release-year bands. Used in the SQ1 group instead of single years: with single years
# only 47% of games sat in groups with >= 10 games on each side; with bands, 81%.
YEAR_BAND_BINS = [0, 2017, 2019, 2021, 2100]
YEAR_BAND_LABELS = ["<=2017", "2018-2019", "2020-2021", "2022+"]
# Starting reach for SQ2 = terciles of reviews in snapshot C. Growth in % depends on the
# starting point (100 -> 200 is +100%, 50,000 -> 55,000 is +10%), so it is held fixed.
REACH_LABELS = ["small", "medium", "large"]

# =============================================================================
# FILTERS AND THRESHOLDS
# =============================================================================
# At least 10 reviews: below that, reach is noise. Not 500: filtering hard on the outcome
# itself would shrink the gap we want to measure.
MIN_REVIEWS = 10
# Released at least one year before the snapshot, so every game had time to collect reviews.
MIN_AGE_DAYS = 365
# A comparison group counts ("eligible") only with >= 10 games on each side; a median of
# fewer games moves too much with a single game.
MIN_GAMES_PER_SIDE = 10
# "Large enough" effects, fixed before looking at results (framework, section 7).
SQ1_THRESHOLD_PCT = 0.20    # SQ1: games with Chinese have >= 20% more reviews
SQ2_THRESHOLD_PTS = 0.10    # SQ2: growth is >= 10 percentage points higher
N_BOOTSTRAP = 2000

# Part A. From the raw files to the analysis tables

## A1. Load the raw snapshots

Download the two snapshots at their pinned commits (skipped when the files are already in
`data/raw/`), then read only the columns we use and give them English snake_case names.

In [ ]:
# LOAD DATA (functions): download the two input CSV files, repair the header, keep the columns we use

def download_snapshots() -> None:
    """Download games.csv of snapshots A and C at the pinned Hugging Face commits."""
    from huggingface_hub import hf_hub_download

    RAW.mkdir(parents=True, exist_ok=True)
    for label, revision in (("a", REVISION_A), ("c", REVISION_C)):
        target = RAW / f"games_{label}.csv"
        if target.exists():
            print(f"snapshot {label.upper()}: already in data/raw")
            continue
        cached = hf_hub_download(repo_id=HF_DATASET, filename=HF_FILENAME,
                                 revision=revision, repo_type="dataset")
        shutil.copyfile(cached, target)
        print(f"snapshot {label.upper()} ({revision[:8]}) downloaded")


def read_header(raw_path) -> list[str]:
    """Read the header row and repair a known defect of this dataset.

    DATA ISSUE: some versions write "DiscountDLC count" as ONE header name while every row
    holds TWO values (discount, DLC count). Without the fix every column after it is
    shifted by one, e.g. "Supported languages" would hold the game description.
    """
    with open(raw_path, newline="", encoding="utf-8") as f:
        header = next(csv.reader(f))
    fixed = []
    for name in header:
        fixed += ["Discount", "DLC count"] if name == "DiscountDLC count" else [name]
    return fixed


def load_snapshot(raw_path) -> pd.DataFrame:
    """Read one raw snapshot: needed columns only, English names, everything as text."""
    names = read_header(raw_path)
    missing = [c for c in RAW_TO_CLEAN if c not in names]
    if missing:
        raise ValueError(f"{raw_path.name}: columns not found: {missing}")
    optional = [c for c in OPTIONAL_RAW_TO_CLEAN if c in names]
    raw = pd.read_csv(
        raw_path, names=names, header=None, skiprows=1,
        usecols=list(RAW_TO_CLEAN) + optional,
        dtype=str, keep_default_na=False, na_values=[""], engine="c",
    )
    # Check that the header repair worked: languages must look like lists.
    langs = raw["Supported languages"].dropna()
    share_list_like = langs.str.startswith("[").mean() if len(langs) else 0
    if share_list_like < 0.9:
        raise ValueError(f"{raw_path.name}: columns look shifted; check read_header()")
    out = raw[list(RAW_TO_CLEAN)].rename(columns=RAW_TO_CLEAN)
    # Optional columns: copy when the snapshot has them, otherwise fill with "0".
    for raw_name, clean_name in OPTIONAL_RAW_TO_CLEAN.items():
        out[clean_name] = raw[raw_name] if raw_name in raw.columns else "0"
    return out

In [ ]:
# LOAD DATA (run): input = 2 raw CSV files, snapshot A (2026-06-20) and C (2024-01-11)
download_snapshots()
raw_a = load_snapshot(RAW / "games_a.csv")
raw_c = load_snapshot(RAW / "games_c.csv")
print(f"snapshot A: {len(raw_a):,} games x {raw_a.shape[1]} columns")
print(f"snapshot C: {len(raw_c):,} games x {raw_c.shape[1]} columns")
raw_a.head(3)

## A2. Clean each snapshot

The same cleaning is applied to A and C: fix types, parse the list-like text columns, and
add every derived column (outcome, treatment, group variables, filter flags). No game is
removed in this step.

In [ ]:
# DATA CLEANING (helpers): clean list-like text cells, pick the genre, split developer names

# DATA ISSUE: language and tag cells contain leftovers from Steam's store pages, such as
# "English[b]*[/b]" (the star marks full audio), HTML tags and line breaks.
_TAG_JUNK = re.compile(r"\[/?[a-zA-Z]+\]|<[^>]+>|&amp;|&lt;|&gt;|\*|\r|\n|#lang_\w+")


def _word_regex(words: list[str]) -> re.Pattern:
    """Case-insensitive whole-word match for any of `words`."""
    return re.compile(r"\b(?:" + "|".join(re.escape(w) for w in words) + r")\b", re.IGNORECASE)


_CN_CITY = _word_regex(CN_CITIES)
_CN_COMPANY = _word_regex(CN_COMPANIES)
_CJK = re.compile(r"[\u4e00-\u9fff]")   # Chinese characters


def parse_list(value) -> list[str]:
    """Turn a list-like cell into a clean list of strings.

    Handles "['English', 'French']", "English,French", stray BBCode such as
    [b]*[/b], HTML leftovers and missing values.
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    text = _TAG_JUNK.sub("", str(value)).strip()
    if not text or text in {"[]", "nan"}:
        return []
    items = None
    if text.startswith("["):
        try:
            items = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            text = text.strip("[]")
    if items is None:
        items = text.split(",")
    cleaned = [str(i).strip().strip("'\"").strip() for i in items]
    seen, out = set(), []
    for item in cleaned:
        if item and item not in seen:
            seen.add(item)
            out.append(item)
    return out


def first_genre(tags: list[str]) -> str | None:
    """First tag (Steam orders tags by votes) that belongs to one of the genre groups."""
    for tag in tags:
        if tag in TAG_TO_GENRE:
            return TAG_TO_GENRE[tag]
    return None


def split_developers(text: str) -> list[str]:
    """Split a developer cell on commas and drop leftover legal suffixes ("Inc.", "Ltd.")."""
    parts = [p.strip().strip(".").strip() for p in str(text).split(",")]
    suffixes = set(COMPANY_SUFFIXES)
    return [p for p in parts if p and p.lower().rstrip(".") not in suffixes]


def all_match(developers: list[str], pattern: re.Pattern) -> bool:
    """True when the game has developers and every one of them matches the pattern."""
    return bool(developers) and all(pattern.search(d) for d in developers)

In [ ]:
# DATA CLEANING + NEW COLUMNS: fix types, parse lists, add outcome, treatment, groups, filter flags
# (no row is removed here)

def add_features(trim: pd.DataFrame, snapshot_date: str) -> pd.DataFrame:
    """Add every derived column to one trimmed snapshot. No row is dropped here
    except rows without a usable app_id."""
    df = trim.copy()

    # --- 1. types: ids and counts to integers, price to float, release date to a date
    # DATA ISSUE: a few rows have no usable id or repeat an id; keep the first occurrence.
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")
    df = df.dropna(subset=["app_id"]).drop_duplicates("app_id")
    df["app_id"] = df["app_id"].astype("int64")

    for col in ("positive", "negative"):
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")
    # Prices in snapshot A are sale prices on the snapshot day (e.g. Hades 7.49 at -70%).
    # Recover the list price so that price tiers describe the game, not the sale.
    df["price_at_snapshot"] = pd.to_numeric(df["price"], errors="coerce")
    discount = df["discount_pct"] if "discount_pct" in df else pd.Series(0, index=df.index)
    df["discount_pct"] = pd.to_numeric(discount, errors="coerce").fillna(0).clip(0, 99)
    df["price"] = (df["price_at_snapshot"] / (1 - df["discount_pct"] / 100)).round(2)
    # DATA ISSUE: dates come as "Aug 1, 2023" or "Aug 2023"; anything unreadable becomes
    # missing and the game is dropped by the "has a release year" filter.
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")
    df["release_year"] = df["release_date"].dt.year.astype("Int64")
    df["days_since_release"] = (pd.Timestamp(snapshot_date) - df["release_date"]).dt.days

    # --- 2. parse the list-like text columns into Python lists
    langs = df["supported_languages"].map(parse_list)
    audio = df["full_audio_languages"].map(parse_list)
    tags = df["tags"].map(parse_list)
    genres = df["genres"].map(parse_list)

    # --- 3. outcome: reach = all reviews, positive or negative
    df["total_reviews"] = df["positive"] + df["negative"]

    # --- 4. language columns: treatment and the "other languages" confounder
    df["has_zh_simplified"] = langs.map(lambda x: ZH_SIMPLIFIED in x)
    df["has_zh_traditional"] = langs.map(lambda x: ZH_TRADITIONAL in x)
    df["has_english"] = langs.map(lambda x: ENGLISH in x)
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    df["n_other_langs"] = langs.map(lambda x: len(set(x) - chinese - {ENGLISH}))
    df["n_other_langs_bin"] = pd.cut(
        df["n_other_langs"], OTHER_LANG_BINS, labels=OTHER_LANG_LABELS
    ).astype(str)
    df["major_langs"] = langs.map(lambda x: "|".join(sorted(set(x) & set(MAJOR_LANGS))))
    df["all_langs"] = langs.map(lambda x: "|".join(sorted(x)))

    # --- 5. Chinese games: ALL are removed (rules and reasons in Settings, "CHINESE GAMES")
    has_zh_audio = audio.map(lambda x: bool(set(x) & chinese))
    has_en_audio = audio.map(lambda x: ENGLISH in x)
    any_zh = df["has_zh_simplified"] | df["has_zh_traditional"]
    developers = df.get("developers", pd.Series("", index=df.index)).fillna("").astype(str)
    publishers = df.get("publishers", pd.Series("", index=df.index)).fillna("").astype(str)
    makers = developers + " | " + publishers
    dev_lists = developers.map(split_developers)
    rules = {
        # rule 1: Chinese text without English text, or Chinese voice-over without English
        "chinese_without_english": (any_zh & ~df["has_english"]) | (has_zh_audio & ~has_en_audio),
        # rules 2-4: EVERY developer is Chinese by that sign (publishers are not used)
        "developer_name_in_chinese": dev_lists.map(lambda d: all_match(d, _CJK)),
        "developer_in_chinese_city": dev_lists.map(lambda d: all_match(d, _CN_CITY)),
        "developer_known_chinese_company": dev_lists.map(lambda d: all_match(d, _CN_COMPANY)),
        # (Python's re is used on purpose instead of .str.contains: with pyarrow installed,
        # pandas switches to Arrow's regex engine, which rejects some of these patterns.)
    }
    rule_frame = pd.DataFrame(rules)
    df["is_cn_game"] = rule_frame.any(axis=1)
    # Which rules matched, e.g. "maker_in_chinese_city;known_chinese_company" (for checking).
    df["cn_rules"] = rule_frame.apply(lambda r: ";".join(r.index[r.to_numpy()]), axis=1)
    df["makers"] = makers
    df["has_zh_audio"] = has_zh_audio
    df["has_english_audio"] = has_en_audio

    # --- 6. other filter flags
    # Early Access games are unfinished; their reviews behave differently.
    # Traditional-only games are neither "with Simplified Chinese" nor "without Chinese".
    df["is_early_access"] = genres.map(lambda x: "Early Access" in x)
    df["is_traditional_only"] = df["has_zh_traditional"] & ~df["has_zh_simplified"]

    # --- 7. grouping variables: genre group, price tier, year band
    df["genre_group"] = tags.map(first_genre)
    df["price_tier"] = pd.cut(df["price"], PRICE_BINS, labels=PRICE_LABELS).astype(str)
    df["year_band"] = pd.cut(
        df["release_year"].astype("float"), YEAR_BAND_BINS, labels=YEAR_BAND_LABELS
    ).astype(str)
    return df

In [ ]:
# DATA CLEANING (run) on both snapshots
feat_a = add_features(raw_a, SNAPSHOT_DATE_A)
feat_c = add_features(raw_c, SNAPSHOT_DATE_C)
print(f"A: {len(feat_a):,} games, C: {len(feat_c):,} games")

## A3. Filters

Filters are applied in order and each one is logged in the funnel table, so the report can
show how many games each rule removed.

In [ ]:
# SELECT ROWS (functions): the filters, applied in order and logged in the funnel

def apply_filters(df: pd.DataFrame, steps: list[tuple[str, pd.Series]], table: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply named filters in order and record how many games survive each one."""
    rows = [{"table": table, "step": "start", "games_left": len(df)}]
    keep = pd.Series(True, index=df.index)
    for name, mask in steps:
        keep &= mask.reindex(df.index).fillna(False)
        rows.append({"table": table, "step": name, "games_left": int(keep.sum())})
    return df[keep].copy(), pd.DataFrame(rows)


def _base_filters(df: pd.DataFrame) -> list[tuple[str, pd.Series]]:
    """Filters shared by games_clean (on A) and panel_ac (on C). Reasons: Settings."""
    return [
        ("drop Chinese games (any of 4 rules)", ~df["is_cn_game"]),
        ("drop Early Access", ~df["is_early_access"]),
        (f"keep >= {MIN_REVIEWS} reviews", df["total_reviews"] >= MIN_REVIEWS),
        ("keep released >= 1 year before snapshot", df["days_since_release"] >= MIN_AGE_DAYS),
        ("keep games with a genre group, price and year",
         df["genre_group"].notna() & df["price"].notna() & df["release_year"].notna()),
    ]

## A4. Table for SQ1: `pcd_sq1_games`

Snapshot A after all filters: one row per game, with the treatment `has_zh`, the outcome
`total_reviews` and the comparison group `group_id`.

In [ ]:
# SELECT ROWS + SELECT COLUMNS -> table games_clean (snapshot A)

def build_games_clean(feat_a: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """One row per game in snapshot A after all filters. Serves EDA and SQ1."""
    steps = _base_filters(feat_a) + [("drop Traditional-Chinese-only", ~feat_a["is_traditional_only"])]
    df, funnel = apply_filters(feat_a, steps, "sq1_games")
    df["has_zh"] = df["has_zh_simplified"].astype(int)
    # Round derived floats so that outputs do not depend on the last bits of a library.
    df["log_total_reviews"] = np.log10(df["total_reviews"]).round(6)
    # Comparison group for SQ1. Release years are merged into bands (allowed by the
    # framework): single years left most groups with < 10 games on one side.
    df["group_id"] = (
        df["genre_group"] + " | " + df["year_band"] + " | "
        + df["price_tier"] + " | " + df["n_other_langs_bin"]
    )
    cols = [
        "app_id", "name", "release_year", "year_band", "price", "discount_pct", "price_tier", "genre_group",
        "n_other_langs", "n_other_langs_bin", "has_zh", "positive", "negative",
        "total_reviews", "log_total_reviews", "group_id",
    ]
    return df[cols].sort_values("app_id").reset_index(drop=True), funnel

In [ ]:
# BUILD TABLE (run): games_clean
games_clean, funnel_a = build_games_clean(feat_a)
print(f"games_clean: {len(games_clean):,} games, {games_clean['has_zh'].sum():,} with Simplified Chinese")
games_clean.head()

## A5. Table for SQ2: `pcd_sq2_games`

Games present in both snapshots that had no Chinese in January 2024 (snapshot C), with
their review growth until snapshot A and whether they added Chinese (`arm`).

In [ ]:
# SELECT ROWS + JOIN A AND C -> table panel_ac: games that could add Chinese, their growth and arm

def build_panel_ac(feat_a: pd.DataFrame, feat_c: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """One row per game present in both snapshots that had no Chinese in C. Serves SQ2.

    Group variables (genre, price, year, other languages) come from C, the starting point,
    because what a game looked like BEFORE adding Chinese is what we hold fixed.
    """
    # Start from games with no Chinese at all in January 2024: only they can "add" Chinese.
    steps = _base_filters(feat_c) + [
        ("keep games with no Chinese (Simplified or Traditional) in C",
         ~(feat_c["has_zh_simplified"] | feat_c["has_zh_traditional"])),
        ("keep games also present in A", feat_c["app_id"].isin(feat_a["app_id"])),
    ]
    c, funnel = apply_filters(feat_c, steps, "sq2_games")
    a = feat_a.set_index("app_id")
    c = c.set_index("app_id")
    a = a.loc[c.index]

    p = pd.DataFrame(index=c.index)
    p["name"] = c["name"]
    p["genre_group"] = c["genre_group"]
    p["release_year"] = c["release_year"]
    p["year_band"] = c["year_band"]
    p["price_c"] = c["price"]  # snapshot C has no discount column: price as listed in C
    p["price_tier"] = c["price_tier"]
    p["n_other_langs_bin"] = c["n_other_langs_bin"]
    p["reviews_c"] = c["total_reviews"]
    p["reviews_a"] = a["total_reviews"]
    # A game with 0 reviews in A had >= 10 in C: it was delisted or its reviews are
    # missing in the data. Its "growth" of -100% is not real, so it is dropped (logged below).
    p = p[p["reviews_a"] > 0].copy()
    a = a.loc[p.index]
    c = c.loc[p.index]
    # Outcome: how much reach grew between the two snapshots.
    p["growth"] = ((p["reviews_a"] - p["reviews_c"]) / p["reviews_c"]).round(6)
    # Starting reach in three equal-size bins (rank first, so ties do not break qcut).
    p["initial_reach_bin"] = pd.qcut(
        p["reviews_c"].rank(method="first"), 3, labels=REACH_LABELS
    ).astype(str)

    # Languages present in A but not in C = languages added after January 2024.
    new_langs = [
        set(la.split("|")) - set(lc.split("|")) - {""}
        for la, lc in zip(a["all_langs"], c["all_langs"])
    ]
    p["added_zh"] = a["has_zh_simplified"].astype(int).to_numpy()
    p["added_only_zh"] = [int(n == {ZH_SIMPLIFIED}) for n in new_langs]
    p["added_other_major_lang"] = [int(bool(n & set(MAJOR_LANGS))) for n in new_langs]
    p["n_langs_added"] = [len(n) for n in new_langs]
    # treated: added Simplified Chinese. control: added neither Chinese nor any major
    # language (a clean "did nothing"). excluded: added another major language only.
    p["arm"] = np.select(
        [p["added_zh"] == 1, (p["added_zh"] == 0) & (p["added_other_major_lang"] == 0)],
        ["treated", "control"],
        default="excluded",
    )
    # Comparison group for SQ2: genre x initial reach. Only ~500 games added Chinese, so the
    # five-way group of the framework left 7 eligible groups; this one keeps ~95% of them
    # and all 6 genres. Year, price and other languages stay in the table for checks.
    p["group_id_sq2"] = p["genre_group"] + " | " + p["initial_reach_bin"]
    funnel = pd.concat([funnel, pd.DataFrame([
        {"table": "sq2_games", "step": "keep games with reviews in A (not delisted)", "games_left": len(p)},
        {"table": "sq2_games", "step": "arm = treated", "games_left": int((p["arm"] == "treated").sum())},
        {"table": "sq2_games", "step": "arm = control", "games_left": int((p["arm"] == "control").sum())},
    ])], ignore_index=True)
    return p.reset_index().sort_values("app_id").reset_index(drop=True), funnel

In [ ]:
# BUILD TABLE (run): panel_ac
panel_ac, funnel_c = build_panel_ac(feat_a, feat_c)
print(f"panel_ac: {len(panel_ac):,} games")
print(panel_ac["arm"].value_counts().to_string())

## A6. What the cleaning removed

Shown here for checking and for the report; not saved as files.
- the funnel: how many games are left after each filter;
- the Chinese games that were removed, with the rule that matched (the biggest first).

In [ ]:
# CHECK: Chinese games that were removed, with the rule that matched

def build_cn_excluded(feat_a: pd.DataFrame) -> pd.DataFrame:
    """Chinese games removed from snapshot A that would otherwise have been kept
    (>= MIN_REVIEWS reviews), with the rule(s) that matched. For checking by hand."""
    df = feat_a[feat_a["is_cn_game"] & (feat_a["total_reviews"] >= MIN_REVIEWS)]
    cols = ["app_id", "name", "makers", "cn_rules", "has_english", "has_zh_audio", "total_reviews"]
    return df[cols].sort_values("total_reviews", ascending=False).reset_index(drop=True)

In [ ]:
# CHECK (run): funnel of both tables, then the 20 biggest Chinese games removed
funnel = pd.concat([funnel_a, funnel_c], ignore_index=True)
cn_excluded = build_cn_excluded(feat_a)
print(f"{len(cn_excluded):,} Chinese games removed (>= {MIN_REVIEWS} reviews)")
display(funnel)
cn_excluded.head(20)

## A7. Save the two tables

Each table is written whole as `data/processed/csv/pcd_<table>.csv` (for code) and
`data/processed/xlsx/pcd_<table>.xlsx` (to view). The xlsx files get fixed document dates
so that their metadata does not change between runs.

In [ ]:
# SAVE OUTPUTS: csv (for code) and xlsx (to view), prefix pcd_
tables = {
    "sq1_games": games_clean,   # EDA, SQ1
    "sq2_games": panel_ac,      # SQ2
}
FIXED_DATE = datetime(2026, 1, 1)
PROCESSED_CSV.mkdir(parents=True, exist_ok=True)
PROCESSED_XLSX.mkdir(parents=True, exist_ok=True)
for name, table in tables.items():
    table.to_csv(PROCESSED_CSV / f"{PREFIX}{name}.csv", index=False)
    with pd.ExcelWriter(PROCESSED_XLSX / f"{PREFIX}{name}.xlsx", engine="openpyxl") as writer:
        table.to_excel(writer, index=False, sheet_name=name[:31])
        writer.book.properties.created = FIXED_DATE
        writer.book.properties.modified = FIXED_DATE
    print(f"{PREFIX}{name}: {len(table):,} rows x {table.shape[1]} columns")

## A8. Checks

These cells stop the notebook with an error if something is wrong.

In [ ]:
# CHECKS: stop with an error if anything is wrong

# Structure: keys are unique, no missing values where groups are formed, English column names.
assert games_clean["app_id"].is_unique and panel_ac["app_id"].is_unique
assert games_clean[["genre_group", "year_band", "price_tier", "n_other_langs_bin"]].notna().all().all()
assert set(games_clean["has_zh"]) <= {0, 1}
assert set(panel_ac["arm"]) <= {"treated", "control", "excluded"}
for name, table in tables.items():
    assert all(re.fullmatch(r"[a-z0-9_]+", c) for c in table.columns), name

# Saved files: csv and xlsx hold the same table.
for name, table in tables.items():
    back = pd.read_excel(PROCESSED_XLSX / f"{PREFIX}{name}.xlsx")
    assert back.shape == table.shape and list(back.columns) == list(table.columns), name

# Known games: data issues and rules behave as intended.
by_name = games_clean.set_index("name")
assert by_name.loc["Hades", "price"] > 20            # list price, not the 7.49 sale price
assert by_name.loc["Hades", "price_tier"] == "gt20"
for name in ["Cyberpunk 2077", "Far Cry® 5", "Core Keeper", "Hollow Knight"]:
    assert name in by_name.index, f"{name} should be kept"          # Western games kept
for name in ["Black Myth: Wukong", "Marvel Rivals", "Dyson Sphere Program"]:
    assert name in set(cn_excluded["name"]), f"{name} should be removed"  # Chinese games removed

# Growth is never -100% (delisted games are dropped).
assert panel_ac["growth"].min() > -1
print("all checks passed")

# Part B. Analysis

Each part starts from the saved tables, so Part B can also be run on its own:
`pcd_sq1_games.csv` for EDA and SQ1, `pcd_sq2_games.csv` for SQ2. Keep figures in
`report/figures/`.

## B1. EDA

In [ ]:
# LOAD TABLES for the analysis
sq1_games = pd.read_csv(PROCESSED_CSV / "pcd_sq1_games.csv")
sq2_games = pd.read_csv(PROCESSED_CSV / "pcd_sq2_games.csv")

## B2. SQ1: how much larger is the reach of games with Simplified Chinese, and in which genres?

First step: compare games with and without Chinese inside each group of similar games.

In [ ]:
# AGGREGATE: one row per group of similar games (medians with vs without Chinese)

def build_sq1_groups(games_clean: pd.DataFrame) -> pd.DataFrame:
    """One row per group: median reviews with vs without Simplified Chinese.

    Games are compared only inside a group of similar games (same genre, year band, price
    tier and number of other languages), so that the gap is not just "bigger games are
    translated more". Medians, not means: review counts are very skewed.
    """
    keys = ["group_id", "genre_group", "year_band", "price_tier", "n_other_langs_bin"]
    wide = (
        games_clean.groupby(keys + ["has_zh"])["total_reviews"]
        .agg(n="size", median="median")
        .unstack("has_zh")
    )
    out = pd.DataFrame({
        "n_zh": wide[("n", 1)] if ("n", 1) in wide else 0,
        "n_no_zh": wide[("n", 0)] if ("n", 0) in wide else 0,
        "median_reviews_zh": wide[("median", 1)] if ("median", 1) in wide else np.nan,
        "median_reviews_no_zh": wide[("median", 0)] if ("median", 0) in wide else np.nan,
    }).reset_index()
    out[["n_zh", "n_no_zh"]] = out[["n_zh", "n_no_zh"]].fillna(0).astype(int)
    # pct_diff: how many % more reviews the games with Chinese have.
    # extra_reviews: that gap in reviews, times the number of games with Chinese.
    out["pct_diff"] = (out["median_reviews_zh"] / out["median_reviews_no_zh"] - 1).round(6)
    out["extra_reviews"] = ((out["median_reviews_zh"] - out["median_reviews_no_zh"]) * out["n_zh"]).round(2)
    out["eligible"] = (out["n_zh"] >= MIN_GAMES_PER_SIDE) & (out["n_no_zh"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values("group_id").reset_index(drop=True)

In [ ]:
# AGGREGATE (run): keep the groups with >= 10 games on each side
sq1_groups = build_sq1_groups(sq1_games)
print(f"{len(sq1_groups)} groups, {sq1_groups['eligible'].sum()} eligible")
sq1_groups[sq1_groups["eligible"]].head()

## B3. SQ2: does adding Simplified Chinese after launch still increase reach, and in which genres?

First step: compare the growth of games that added Chinese (treated) with similar games
that did not (control), per genre and starting reach.

In [ ]:
# AGGREGATE: treated vs control growth per genre x starting reach

def build_sq2_groups(panel: pd.DataFrame) -> pd.DataFrame:
    """One row per SQ2 group: median growth of treated vs control games.

    uplift_pts = extra growth of games that added Chinese, compared with similar games that
    did not. The control arm is needed because every game gains reviews over time anyway.
    """
    keys = ["group_id_sq2", "genre_group", "initial_reach_bin"]
    used = panel[panel["arm"].isin(["treated", "control"])]
    wide = used.groupby(keys + ["arm"])["growth"].agg(n="size", median="median").unstack("arm")

    def col(stat, arm, default):
        return wide[(stat, arm)] if (stat, arm) in wide else default

    out = pd.DataFrame({
        "n_treated": col("n", "treated", 0),
        "n_control": col("n", "control", 0),
        "median_growth_treated": col("median", "treated", np.nan),
        "median_growth_control": col("median", "control", np.nan),
    }).reset_index()
    out[["n_treated", "n_control"]] = out[["n_treated", "n_control"]].fillna(0).astype(int)
    out[["median_growth_treated", "median_growth_control"]] = out[
        ["median_growth_treated", "median_growth_control"]
    ].round(6)
    out["uplift_pts"] = (out["median_growth_treated"] - out["median_growth_control"]).round(6)
    out["eligible"] = (out["n_treated"] >= MIN_GAMES_PER_SIDE) & (out["n_control"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values("group_id_sq2").reset_index(drop=True)

In [ ]:
# AGGREGATE (run)
sq2_groups = build_sq2_groups(sq2_games)
sq2_groups

## B4. SQ3: for each genre, translate at launch, later, or not a priority?